# **AI로 웹데이터 수집 및 시각화**



---



## **1-4.동적 크롤링**(**Visual Studio Code 사용**)

- 공식페이지: https://www.selenium.dev/
- 참고: https://wikidocs.net/198942
- beautifulsoup 사용법 : https://wikidocs.net/85739
    - **select_one** 은 찾은 html 중 가장 첫번째 html 을 가져오고
    - **select** 는 찾은 모든 html 을 리스트 형태로 반환
- 크롬 브라우저 필요
- ※ 동적 크롤링은 PC에서 실행하세요 (IDLE 또는 Visual Studio Code)


* **라이브러리 설치하기**

In [5]:
%pip install pandas numpy matplotlib seaborn scikit-learn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --------- ------------------------------ 2.4/10.0 MB 11.2 MB/s eta 0:00:01
   ------------------ --------------------- 4.7/10.0 MB 11.1 MB/s eta 0:00:01
   ---------------------------- ----------- 7.1/10.0 MB 11.2 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.0 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 10.7 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ------- -------------------------------- 2.4/12.6 MB 11.9 MB/s eta 0:00:01
   --------------- ------------------------ 5.0/12.6 MB 12.0 MB/s eta 0:00:01
   ----------------------- ---------------- 7.3/12.6 MB 12.0 MB/s eta 0:00:01
   ------------------------------- -------- 10.0/12.6 MB 11.8 MB/s eta 0:00:01
   ---------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [6]:
%pip install requests beautifulsoup4 selenium chromedriver-autoinstaller

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.5 MB 5.1 MB/s eta 0:00:02
   ------------- -------------------------- 3.1/9.5 MB 7.3 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.5 MB 8.8 MB/s eta 0:00:01
   ---------------------------------- ----- 8.1/9.5 MB 9.7 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 9.3 MB/s  0:00:01

   ----------------------------------------  0/21 [sortedcontainers]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
   - --------------------------------------  1/21 [websocket-client]
 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


- **한글 폰트 지정하기**

In [7]:
# 코랩에서 한글 폰트 종류와 이름이 win과 다를 수 있다!!!
# 코랩: NanumGothic, 윈도우: Malgun Gothic
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'Malgun Gothic',
                     'font.size': 12,
                     'figure.figsize': (6, 4),
                     'axes.unicode_minus':  False }) # 폰트 설정

Matplotlib is building the font cache; this may take a moment.


In [1]:
import selenium
selenium.__version__

'4.47.0'

### 1️⃣ 웹 드라이브 테스트

In [2]:
from selenium import webdriver

# 드라이버 초기화
driver = webdriver.Chrome()

# 웹페이지로 이동
driver.get('https://www.weather.go.kr')

# 브라우저 탭 닫기
driver.close()

# 브라우저 종료하기 (탭 모두 종료)
driver.quit()

* **Target 웹 페이지**

In [3]:
# ============================================================
# 동적 크롤링 — 기상청 시간별 예보 수집하기 (Selenium)
# 대상 페이지 : https://www.weather.go.kr/w/weather/forecast/short-term.do
# ============================================================
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

URL = 'https://www.weather.go.kr/w/weather/forecast/short-term.do'
FILE = os.path.abspath('시간별예보.csv')   # 저장 위치를 '절대경로'로 고정한다

# 표로 만들 항목 — 여기만 고치면 항목을 자유롭게 넣고 뺄 수 있다.
COLUMNS = ['시각', '날씨', '기온', '체감온도', '강수확률', '습도']

# 1) 크롬을 자동 실행하고 단기예보 페이지 열기 ---------------------
driver = webdriver.Chrome()

try:
    driver.get(URL)

    # 2) 시간별 예보가 "로딩될 때까지" 기다리기 (동적 크롤링의 핵심!) --
    #    requests로 받으면 '로딩중...'만 보이지만,
    #    Selenium은 자바스크립트 실행이 끝난 화면을 읽을 수 있다.
    #    ※ 빈 표 껍데기가 먼저 그려지므로, '시각' 글자가 들어올 때까지 기다린다.
    wait = WebDriverWait(driver, 30)          # 인터넷이 느려도 되도록 30초로 넉넉하게
    wait.until(lambda d: any(
        '시각' in (li.get_attribute('textContent') or '')
        for li in d.find_elements(By.CSS_SELECTOR, '#digital-forecast ul.item.s-item li')))
    print('시간별 예보 로딩 완료!')
    

    # 3) 시간대별 항목 추출 --------------------------------------------
    rows = []
    for item in driver.find_elements(By.CSS_SELECTOR, '#digital-forecast ul.item.s-item'):
        info = {}
        for li in item.find_elements(By.TAG_NAME, 'li'):
            text = (li.get_attribute('textContent') or '').replace('\xa0', ' ').strip()
            if ':' in text:
                key, value = text.split(':', 1)      # '시각: 15시' → 시각 / 15시
                info[key.strip()] = value.strip()    # '기온 :' 처럼 공백이 있어도 strip으로 정리
        if info:                                     # 값이 하나도 없는 껍데기 <ul>은 건너뛴다
            info['기온'] = info.get('기온', '').replace('℃', '')
            info['체감온도'] = info.get('체감온도', '').replace('℃', '')
            rows.append(info)

finally:
    driver.quit()          # 오류가 나도 크롬은 반드시 닫는다(좀비 크롬 방지)

# 어떤 항목들을 읽어왔는지 확인 (페이지 구조가 바뀌면 여기서 바로 보인다)
if not rows:
    raise RuntimeError('한 건도 수집되지 않았습니다. F12로 선택자를 다시 확인하세요.')
print('읽어온 항목 :', list(rows[0].keys()))

# 4) DataFrame으로 정리하고 CSV로 저장 -----------------------------
#    columns= 로 원하는 항목만, 원하는 순서로 골라 담는다.
df = pd.DataFrame(rows, columns=COLUMNS)
print(df.head(8))
print(f'수집 완료 : {len(df)}개 시간대')

df.to_csv(FILE, index=False, encoding='utf-8-sig')
print(f'저장 완료 : {FILE}')          # ← 어느 폴더에 저장됐는지 꼭 확인!

# ------------------------------------------------------------------
# ★ 도전 1 : '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를
#            수집해 보세요. (28개 → 54개 시간대)
#            버튼 선택자 : .tab-btn-wrap a.tab-btn  (F12로 확인)
#
# import time
# for btn in driver.find_elements(By.CSS_SELECTOR, '.tab-btn-wrap a.tab-btn'):
#     if btn.text.strip() == '1시간 간격':
#         btn.click()
#         time.sleep(2)        # 데이터가 다시 그려질 때까지 잠시 대기
#         break
#
# ★ 도전 2 : COLUMNS 에 '바람', '폭염영향' 을 추가해서 함께 저장해 보세요.
#            (위 '읽어온 항목' 출력에 나오는 이름을 그대로 넣으면 된다)
#
# ★ 도전 3 : 수집한 시간별 기온으로 꺾은선 그래프를 그려 보세요.
# import matplotlib.pyplot as plt
# plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.plot(df['시각'], pd.to_numeric(df['기온']), marker='o', color='#0E9CA0')
# plt.title('시간별 기온 예보'); plt.show()
# ------------------------------------------------------------------


시간별 예보 로딩 완료!
읽어온 항목 : ['시각', '날씨', '기온', '체감온도', '강수량', '강수확률', '바람', '습도', '폭염영향']
    시각     날씨  기온 체감온도 강수확률   습도
0  12시     맑음  32   32   0%  60%
1  15시     맑음  32   33   0%  70%
2  18시  구름 많음  30   32  20%  75%
3  21시     흐림  28   30  30%  80%
4   0시  구름 많음  28   30  20%  85%
5  03시  구름 많음  27   29  20%  85%
6  06시  구름 많음  27   29  20%  85%
7  09시  구름 많음  30   32  20%  75%
수집 완료 : 29개 시간대
저장 완료 : c:\Users\이주희\Downloads\시간별예보.csv


### **[실습] '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를 자동 수집하기.**

#### 프롬프트
>  앞에서 사용한 수집 코드를 참고하여 화면에서 '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보를 자동 수집하는 코드 만들어줘.

In [4]:
# ============================================================
# '1시간 간격' 버튼을 클릭해서 더 촘촘한 예보 수집하기
#  - 앞의 수집 코드에 "버튼 클릭" 한 단계를 더한 것이다.
#  - 3시간 간격(32개) → 1시간 간격(66개)으로 늘어난다.
# ============================================================
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

URL = 'https://www.weather.go.kr/w/weather/forecast/short-term.do'
FILE = os.path.abspath('시간별예보_1시간간격.csv')
COLUMNS = ['시각', '날씨', '기온', '체감온도', '강수확률', '습도']

ITEM = '#digital-forecast ul.item.s-item'                       # 시간대 한 칸
BTN_1H = '.tab-btn-wrap a.tab-btn[data-interval-hours="1"]'     # '1시간 간격' 버튼


# [함수] 화면에 그려진 시간대 칸들을 읽어서 리스트로 돌려준다 -------
#        (실습 3의 추출 부분을 함수로 묶은 것이다)
def read_forecast(driver):
    rows = []
    for item in driver.find_elements(By.CSS_SELECTOR, ITEM):
        info = {}
        for li in item.find_elements(By.TAG_NAME, 'li'):
            # 라벨 span은 class="hid"로 숨겨져 있으므로 반드시 textContent로 읽는다
            text = (li.get_attribute('textContent') or '').replace('\xa0', ' ').strip()
            if ':' in text:
                key, value = text.split(':', 1)      # '시각: 15시' → 시각 / 15시
                info[key.strip()] = value.strip()

        # ★ 기온은 따로 챙긴다 ★
        #   3시간 간격 : <span class="hid">기온 : </span>        ← ':' 가 있어서 위 반복문에 잡힌다
        #   1시간 간격 : <span class="hid">기온(체감온도) </span>  ← ':' 가 없어서 안 잡힌다!
        #   → 두 화면 모두에 있는 <span class="feel"> 에서 직접 읽으면 안전하다.
        #   1시간 간격의 feel 값은 '25℃(28℃)' 처럼 괄호가 붙으므로 앞부분만 쓴다.
        feel = item.find_elements(By.CSS_SELECTOR, 'span.feel')
        if feel:
            info['기온'] = (feel[0].get_attribute('textContent') or '').split('(')[0]

        if info:                                     # 값이 하나도 없는 껍데기 <ul>은 건너뛴다
            info['기온'] = info.get('기온', '').replace('℃', '').strip()
            info['체감온도'] = info.get('체감온도', '').replace('℃', '').strip()
            rows.append(info)
    return rows


driver = webdriver.Chrome()

try:
    driver.get(URL)

    # 1) 먼저 기본 화면(3시간 간격)이 다 그려질 때까지 기다린다 --------
    wait = WebDriverWait(driver, 30)
    wait.until(lambda d: any(
        '시각' in (li.get_attribute('textContent') or '')
        for li in d.find_elements(By.CSS_SELECTOR, ITEM + ' li')))
    before = len(driver.find_elements(By.CSS_SELECTOR, ITEM))
    print(f'3시간 간격 : {before}개 시간대')

    # 2) '1시간 간격' 버튼 클릭 ★동적 크롤링의 핵심★ ------------------
    #    execute_script로 누르면 버튼이 화면 아래쪽에 있어도 확실하게 눌린다.
    button = driver.find_element(By.CSS_SELECTOR, BTN_1H)
    driver.execute_script('arguments[0].click();', button)
    print("'1시간 간격' 버튼 클릭!")


    # 3) 표가 "다시 그려질 때까지" 기다린다 ---------------------------
    #    time.sleep(2)로 무작정 기다리지 말고,
    #    칸 수가 늘어날 때까지 기다리는 것이 훨씬 정확하다.
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, ITEM)) > before)
    print(f'1시간 간격 : {len(driver.find_elements(By.CSS_SELECTOR, ITEM))}개 시간대로 늘어남')


    # 4) 촘촘해진 예보 읽어오기 ---------------------------------------
    rows = read_forecast(driver)

finally:
    driver.quit()          # 오류가 나도 크롬은 반드시 닫는다(좀비 크롬 방지)


# 5) DataFrame으로 정리하고 CSV로 저장 -----------------------------
if not rows:
    raise RuntimeError('한 건도 수집되지 않았습니다. F12로 선택자를 다시 확인하세요.')
print('읽어온 항목 :', list(rows[0].keys()))

df = pd.DataFrame(rows, columns=COLUMNS)
print(df.head(10))
print(f'수집 완료 : {len(df)}개 시간대')

df.to_csv(FILE, index=False, encoding='utf-8-sig')
print(f'저장 완료 : {FILE}')

3시간 간격 : 29개 시간대
'1시간 간격' 버튼 클릭!
1시간 간격 : 57개 시간대로 늘어남


StaleElementReferenceException: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=151.0.7922.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff704ffabb5+151b5]
	chromedriver!GetHandleVerifier [0x7ff704ffac10+15210]
	chromedriver!(No symbol) [0x7ff704b25d6d]
	chromedriver!(No symbol) [0x7ff704b2dcd1]
	chromedriver!(No symbol) [0x7ff704b3101f]
	chromedriver!(No symbol) [0x7ff704bcfe82]
	chromedriver!(No symbol) [0x7ff704ba9d0a]
	chromedriver!(No symbol) [0x7ff704bcec0b]
	chromedriver!(No symbol) [0x7ff704b737cc]
	chromedriver!(No symbol) [0x7ff704b746f3]
	chromedriver!GetHandleVerifier [0x7ff70561b79b+635d9b]
	chromedriver!GetHandleVerifier [0x7ff705615ac6+6300c6]
	chromedriver!GetHandleVerifier [0x7ff70563bcee+6562ee]
	chromedriver!GetHandleVerifier [0x7ff705018ffe+335fe]
	chromedr

--------------

In [8]:
%pip install selenium webdriver-manager pandas -q

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import time


# ==========================================
# 1. Chrome 설정
# ==========================================

options = Options()

# Colab에서 Chrome을 화면 없이 실행
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 10)


# ==========================================
# 2. 네이버 접속
# ==========================================

driver.get("https://www.naver.com/")

time.sleep(2)


# ==========================================
# 3. 검색어 입력
# ==========================================

search_box = wait.until(
    EC.presence_of_element_located((By.NAME, "query"))
)

search_box.send_keys("노벨문학상")
search_box.send_keys(Keys.ENTER)

time.sleep(3)


# ==========================================
# 4. 뉴스 검색 결과로 이동
# ==========================================

# 검색 결과에서 뉴스 탭 클릭
try:
    news_tab = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//a[contains(., '뉴스')]")
        )
    )
    news_tab.click()

except:
    # 뉴스 탭을 직접 URL로 이동
    driver.get(
        "https://search.naver.com/search.naver"
        "?where=news"
        "&query=노벨문학상"
    )

time.sleep(3)


# ==========================================
# 5. 최신순 정렬
# ==========================================

try:

    # 최신순 버튼 찾기
    latest_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//a[contains(., '최신순')]")
        )
    )

    latest_button.click()

except:
    print("최신순 버튼을 찾지 못했습니다.")
    print("현재 페이지를 계속 크롤링합니다.")


time.sleep(3)


# ==========================================
# 6. 뉴스 데이터 수집
# ==========================================

news_data = []

while len(news_data) < 100:

    # 뉴스 기사 목록
    articles = driver.find_elements(
        By.CSS_SELECTOR,
        "div.sds-comps-vertical-layout"
    )

    # 기존 선택자가 변경되었을 경우 사용할 후보
    if not articles:
        articles = driver.find_elements(
            By.CSS_SELECTOR,
            "div.news_area"
        )

    for article in articles:

        if len(news_data) >= 100:
            break

        try:

            # 제목
            title_element = article.find_element(
                By.CSS_SELECTOR,
                "a.news_tit"
            )

            title = title_element.text.strip()

            # 기사 URL
            url = title_element.get_attribute("href")

            # 언론사
            try:
                press = article.find_element(
                    By.CSS_SELECTOR,
                    ".info_group .info"
                ).text.strip()
            except:
                press = ""

            # 날짜
            try:
                date_elements = article.find_elements(
                    By.CSS_SELECTOR,
                    ".info_group .info"
                )

                date = ""

                for element in date_elements:
                    text = element.text.strip()

                    if "전" in text or "." in text or "-" in text:
                        date = text
                        break

            except:
                date = ""

            # 기사 요약
            try:
                description = article.find_element(
                    By.CSS_SELECTOR,
                    ".news_dsc"
                ).text.strip()
            except:
                description = ""

            # 중복 기사 제거
            if title and not any(
                x["제목"] == title for x in news_data
            ):

                news_data.append({
                    "순위": len(news_data) + 1,
                    "제목": title,
                    "언론사": press,
                    "날짜": date,
                    "요약": description,
                    "URL": url
                })

        except:
            continue


    # ==========================================
    # 7. 다음 페이지 이동
    # ==========================================

    if len(news_data) < 100:

        try:

            next_button = driver.find_element(
                By.CSS_SELECTOR,
                "a.btn_next"
            )

            driver.execute_script(
                "arguments[0].click();",
                next_button
            )

            time.sleep(2)

        except:

            # 다음 페이지가 없으면 종료
            print("다음 페이지를 찾을 수 없습니다.")
            break


# ==========================================
# 8. 상위 100건만 추출
# ==========================================

news_data = news_data[:100]

df = pd.DataFrame(news_data)


# ==========================================
# 9. 결과 출력
# ==========================================

print(f"총 수집 기사 수: {len(df)}")

display(df)


# ==========================================
# 10. CSV 파일 저장
# ==========================================

df.to_csv(
    "naver_news_nobel_literature_100.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV 저장 완료")


# ==========================================
# 11. 브라우저 종료
# ==========================================

driver.quit()

다음 페이지를 찾을 수 없습니다.
총 수집 기사 수: 0


""


CSV 저장 완료


---------------

In [10]:
pip install selenium pandas openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   ---------------------------------------- 0/2 [et-xmlfile]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ----------------

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import time


# ============================================================
# 기본 설정
# ============================================================

KEYWORD = "노벨문학상"
TARGET_COUNT = 100

BASE_URL = "https://search.naver.com/search.naver"


# ============================================================
# Chrome 설정
# ============================================================

options = Options()

# 브라우저 화면을 직접 확인하고 싶으면 아래 두 줄을 주석 처리
# options.add_argument("--headless=new")

options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 10)


# ============================================================
# 네이버 뉴스 검색
# ============================================================

search_url = (
    f"{BASE_URL}"
    f"?where=news"
    f"&query={KEYWORD}"
    f"&sort=1"
)

print("=" * 60)
print("네이버 뉴스 크롤링 시작")
print("검색어 :", KEYWORD)
print("목표 기사 :", TARGET_COUNT)
print("=" * 60)

driver.get(search_url)

time.sleep(3)


# ============================================================
# 뉴스 검색 결과 확인
# ============================================================

print("\n현재 페이지:", driver.title)


# 뉴스 제목 요소가 나타날 때까지 기다림
try:

    wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, "a.news_tit")
        )
    )

except Exception:

    print("뉴스 검색 결과를 찾지 못했습니다.")
    print("현재 URL:", driver.current_url)

    driver.quit()
    exit()


# ============================================================
# 뉴스 크롤링
# ============================================================

news_data = []

page = 1


while len(news_data) < TARGET_COUNT:

    print(f"\n[{page}페이지] 크롤링 중...")


    # --------------------------------------------------------
    # 현재 페이지의 뉴스 제목
    # --------------------------------------------------------

    articles = driver.find_elements(
        By.CSS_SELECTOR,
        "a.news_tit"
    )

    print("발견한 뉴스:", len(articles))


    if len(articles) == 0:

        print("뉴스 기사를 찾지 못했습니다.")
        break


    # --------------------------------------------------------
    # 뉴스 하나씩 처리
    # --------------------------------------------------------

    for article in articles:

        if len(news_data) >= TARGET_COUNT:
            break

        try:

            title = article.text.strip()

            url = article.get_attribute("href")


            # 제목이 없는 경우 제외
            if not title:
                continue


            # 중복 제거
            if any(
                item["제목"] == title
                for item in news_data
            ):
                continue


            # ------------------------------------------------
            # 기사 전체 영역
            # ------------------------------------------------

            try:

                news_area = article.find_element(
                    By.XPATH,
                    "./ancestor::div[contains(@class,'news_area')]"
                )

            except:

                news_area = article


            # ------------------------------------------------
            # 언론사
            # ------------------------------------------------

            try:

                info_elements = news_area.find_elements(
                    By.CSS_SELECTOR,
                    ".info_group .info"
                )

                press = ""

                for info in info_elements:

                    text = info.text.strip()

                    if text and "전" not in text:

                        press = text
                        break

            except:

                press = ""


            # ------------------------------------------------
            # 날짜
            # ------------------------------------------------

            try:

                info_elements = news_area.find_elements(
                    By.CSS_SELECTOR,
                    ".info_group .info"
                )

                date = ""

                for info in info_elements:

                    text = info.text.strip()

                    if (
                        "전" in text
                        or "." in text
                        or "-" in text
                    ):

                        date = text

            except:

                date = ""


            # ------------------------------------------------
            # 기사 요약
            # ------------------------------------------------

            try:

                description = news_area.find_element(
                    By.CSS_SELECTOR,
                    ".news_dsc"
                ).text.strip()

            except:

                description = ""


            # ------------------------------------------------
            # 저장
            # ------------------------------------------------

            news_data.append({

                "순위": len(news_data) + 1,

                "제목": title,

                "언론사": press,

                "날짜": date,

                "요약": description,

                "URL": url

            })


            print(
                f"{len(news_data):3d}. {title}"
            )


        except Exception:

            continue


    # ========================================================
    # 100건 수집 완료
    # ========================================================

    if len(news_data) >= TARGET_COUNT:

        print("\n100건 수집 완료!")
        break


    # ========================================================
    # 다음 페이지
    #
    # 네이버 뉴스 검색 페이지 번호를 직접 변경
    # ========================================================

    page += 1

    start = (page - 1) * 10 + 1

    next_url = (
        f"{BASE_URL}"
        f"?where=news"
        f"&query={KEYWORD}"
        f"&sort=1"
        f"&start={start}"
    )

    print(f"→ {page}페이지 이동")

    driver.get(next_url)

    time.sleep(2)


# ============================================================
# DataFrame 생성
# ============================================================

df = pd.DataFrame(news_data)


# ============================================================
# 결과 확인
# ============================================================

print("\n")
print("=" * 60)
print("크롤링 완료")
print("=" * 60)

print("총 수집 기사:", len(df))


if len(df) > 0:

    print("\n[수집 결과]")
    print(df.head(10).to_string(index=False))


# ============================================================
# CSV 저장
# ============================================================

csv_file = "naver_news_nobel_literature_100.csv"

df.to_csv(
    csv_file,
    index=False,
    encoding="utf-8-sig"
)

print("\nCSV 저장 완료:")
print(csv_file)


# ============================================================
# Excel 저장
# ============================================================

excel_file = "naver_news_nobel_literature_100.xlsx"

df.to_excel(
    excel_file,
    index=False
)

print("\nExcel 저장 완료:")
print(excel_file)


# ============================================================
# 브라우저 종료
# ============================================================

driver.quit()

print("\n프로그램 종료")

네이버 뉴스 크롤링 시작
검색어 : 노벨문학상
목표 기사 : 100

현재 페이지: 노벨문학상 : 네이버 뉴스검색
뉴스 검색 결과를 찾지 못했습니다.


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=151.0.7922.138)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff704ffabb5+151b5]
	chromedriver!GetHandleVerifier [0x7ff704ffac10+15210]
	chromedriver!(No symbol) [0x7ff704b25d6d]
	chromedriver!(No symbol) [0x7ff704afcc52]
	chromedriver!(No symbol) [0x7ff704bb10c6]
	chromedriver!(No symbol) [0x7ff704bce4f2]
	chromedriver!(No symbol) [0x7ff704b737cc]
	chromedriver!(No symbol) [0x7ff704b746f3]
	chromedriver!GetHandleVerifier [0x7ff70561b79b+635d9b]
	chromedriver!GetHandleVerifier [0x7ff705615ac6+6300c6]
	chromedriver!GetHandleVerifier [0x7ff70563bcee+6562ee]
	chromedriver!GetHandleVerifier [0x7ff705018ffe+335fe]
	chromedriver!GetHandleVerifier [0x7ff70502147c+3ba7c]
	chromedriver!GetHandleVerifier [0x7ff705004d64+1f364]
	chromedriver!GetHandleVerifier [0x7ff705004ef4+1f4f4]
	chromedriver!GetHandleVerifier [0x7ff704fe7d97+2397]
	KERNEL32!BaseThreadIn

In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

options = Options()

# 브라우저를 실제로 띄움
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

print("Chrome 실행 성공!")
print("현재 URL:", driver.current_url)

driver.get("https://www.naver.com/")

time.sleep(5)

print("네이버 접속 성공!")
print("페이지 제목:", driver.title)
print("현재 URL:", driver.current_url)

input("Chrome을 확인한 후 Enter를 누르세요...")

driver.quit()

print("Chrome 종료 완료")

Chrome 실행 성공!
현재 URL: data:,
네이버 접속 성공!
페이지 제목: NAVER
현재 URL: https://www.naver.com/
Chrome 종료 완료


In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import time
import urllib.parse


# ============================================================
# 기본 설정
# ============================================================

KEYWORD = "노벨문학상"
TARGET_COUNT = 100


# ============================================================
# Chrome 실행
# ============================================================

options = webdriver.ChromeOptions()

# 브라우저 화면을 직접 확인하기 위해 headless 사용 안 함
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 15)


# ============================================================
# 1. 네이버 접속
# ============================================================

print("=" * 60)
print("1. 네이버 접속")
print("=" * 60)

driver.get("https://www.naver.com/")

time.sleep(2)

print("네이버 접속 완료")
print("페이지 제목:", driver.title)


# ============================================================
# 2. 검색창 찾기
# ============================================================

print("\n2. 검색어 입력")

try:

    search_box = wait.until(
        EC.presence_of_element_located(
            (By.NAME, "query")
        )
    )

    search_box.click()
    search_box.send_keys(KEYWORD)
    search_box.send_keys(Keys.ENTER)

    time.sleep(3)

    print("검색어 입력 완료:", KEYWORD)

except Exception as e:

    print("검색창을 찾지 못했습니다.")
    print(e)

    driver.quit()
    raise


# ============================================================
# 3. 현재 페이지 확인
# ============================================================

print("\n현재 URL:")
print(driver.current_url)

print("\n현재 페이지 제목:")
print(driver.title)


# ============================================================
# 4. 뉴스 탭 찾기
# ============================================================

print("\n3. 뉴스 탭 찾는 중...")


news_clicked = False


# 방법 1 : 뉴스라는 글자가 있는 링크 찾기
try:

    news_links = driver.find_elements(
        By.XPATH,
        "//a[contains(normalize-space(.), '뉴스')]"
    )

    print("뉴스 관련 링크:", len(news_links))

    for link in news_links:

        try:

            text = link.text.strip()

            if text == "뉴스" or "뉴스" in text:

                driver.execute_script(
                    "arguments[0].click();",
                    link
                )

                news_clicked = True

                print("뉴스 탭 클릭 성공")
                break

        except:
            continue

except Exception as e:

    print("뉴스 탭 탐색 실패:", e)


# ============================================================
# 뉴스 탭 클릭 실패하면 뉴스 검색 URL 직접 접속
# ============================================================

if not news_clicked:

    print("뉴스 탭을 찾지 못했습니다.")
    print("뉴스 검색 URL로 직접 이동합니다.")

    encoded_keyword = urllib.parse.quote(KEYWORD)

    news_url = (
        "https://search.naver.com/search.naver"
        "?where=news"
        f"&query={encoded_keyword}"
    )

    driver.get(news_url)


# 페이지 로딩
time.sleep(4)


# ============================================================
# 5. 뉴스 페이지 확인
# ============================================================

print("\n4. 뉴스 검색 페이지 확인")

print("현재 URL:")
print(driver.current_url)

print("페이지 제목:")
print(driver.title)


# ============================================================
# 6. 뉴스 제목 찾기
# ============================================================

print("\n5. 뉴스 기사 찾는 중...")


# 여러 선택자를 순서대로 시도
selectors = [

    "a.news_tit",

    "a[class*='news_tit']",

    "div.news_area a",

    "div[class*='news_area'] a",

]


articles = []


for selector in selectors:

    try:

        found = driver.find_elements(
            By.CSS_SELECTOR,
            selector
        )

        print(
            f"선택자 {selector} → {len(found)}개"
        )

        if len(found) > 0:

            articles = found
            break

    except Exception as e:

        print("선택자 오류:", e)


# ============================================================
# 결과 확인
# ============================================================

if len(articles) == 0:

    print("\n❌ 뉴스 기사를 찾지 못했습니다.")

    print("\n현재 페이지의 링크 일부를 확인합니다.")

    links = driver.find_elements(
        By.TAG_NAME,
        "a"
    )

    for link in links[:30]:

        try:

            text = link.text.strip()

            if text:

                print(
                    "LINK:",
                    text[:100]
                )

        except:
            pass

    input(
        "\n브라우저 화면을 확인한 후 "
        "Enter를 누르세요..."
    )

    driver.quit()

    raise Exception(
        "네이버 뉴스 기사 요소를 찾지 못했습니다."
    )


# ============================================================
# 7. 뉴스 기사 제목 출력
# ============================================================

print("\n" + "=" * 60)
print("뉴스 기사 발견!")
print("=" * 60)

print("현재 페이지에서 찾은 기사:", len(articles))


for i, article in enumerate(
    articles[:10],
    start=1
):

    try:

        title = article.text.strip()
        url = article.get_attribute("href")

        print()
        print(i, ".", title)
        print("URL:", url)

    except:
        pass


# ============================================================
# 8. 테스트 완료
# ============================================================

print("\n" + "=" * 60)
print("뉴스 검색 테스트 성공")
print("=" * 60)

input(
    "\nChrome 화면을 확인했다면 Enter를 누르세요..."
)

driver.quit()

1. 네이버 접속
네이버 접속 완료
페이지 제목: NAVER

2. 검색어 입력
검색어 입력 완료: 노벨문학상

현재 URL:
https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%EB%85%B8%EB%B2%A8%EB%AC%B8%ED%95%99%EC%83%81&ackey=ghgg1a8a

현재 페이지 제목:
노벨문학상 : 네이버 검색

3. 뉴스 탭 찾는 중...
뉴스 관련 링크: 7
뉴스 탭 클릭 성공

4. 뉴스 검색 페이지 확인
현재 URL:


InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=151.0.7922.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff704ffabb5+151b5]
	chromedriver!GetHandleVerifier [0x7ff704ffac10+15210]
	chromedriver!(No symbol) [0x7ff704b25d6d]
	chromedriver!(No symbol) [0x7ff704b115c2]
	chromedriver!(No symbol) [0x7ff704b374a1]
	chromedriver!(No symbol) [0x7ff704bb1320]
	chromedriver!(No symbol) [0x7ff704bce4f2]
	chromedriver!(No symbol) [0x7ff704b737cc]
	chromedriver!(No symbol) [0x7ff704b746f3]
	chromedriver!GetHandleVerifier [0x7ff70561b79b+635d9b]
	chromedriver!GetHandleVerifier [0x7ff705615ac6+6300c6]
	chromedriver!GetHandleVerifier [0x7ff70563bcee+6562ee]
	chromedriver!GetHandleVerifier [0x7ff705018ffe+335fe]
	chromedri

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import time
import urllib.parse


# ============================================================
# 1. 새로운 Selenium 세션 생성
# ============================================================

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 15)

print("Chrome 실행 성공!")


# ============================================================
# 2. 네이버 접속
# ============================================================

driver.get("https://www.naver.com/")

time.sleep(3)

print("네이버 접속 성공!")
print("페이지 제목:", driver.title)
print("현재 URL:", driver.current_url)


# ============================================================
# 3. 검색창에 노벨문학상 입력
# ============================================================

search_box = wait.until(
    EC.presence_of_element_located(
        (By.NAME, "query")
    )
)

search_box.click()
search_box.send_keys("노벨문학상")
search_box.send_keys(Keys.ENTER)

time.sleep(4)

print("\n검색 완료")
print("현재 URL:", driver.current_url)
print("페이지 제목:", driver.title)


# ============================================================
# 4. 뉴스 검색 결과로 이동
# ============================================================

keyword = urllib.parse.quote("노벨문학상")

news_url = (
    "https://search.naver.com/search.naver"
    f"?where=news&query={keyword}"
)

print("\n뉴스 검색 페이지로 이동합니다.")

driver.get(news_url)

time.sleep(5)

print("뉴스 페이지 접속 완료")
print("현재 URL:", driver.current_url)
print("페이지 제목:", driver.title)


# ============================================================
# 5. 뉴스 기사 찾기
# ============================================================

articles = driver.find_elements(
    By.CSS_SELECTOR,
    "a.news_tit"
)

print("\n찾은 뉴스 기사 수:", len(articles))


# ============================================================
# 6. 뉴스 제목 확인
# ============================================================

if len(articles) > 0:

    print("\n===== 뉴스 기사 =====")

    for i, article in enumerate(articles[:10], 1):

        title = article.text.strip()
        url = article.get_attribute("href")

        print(f"\n{i}. {title}")
        print(f"   {url}")

else:

    print("\n뉴스 기사를 찾지 못했습니다.")

    # 현재 페이지의 a 태그 중 텍스트가 있는 것 확인
    links = driver.find_elements(By.TAG_NAME, "a")

    print("\n현재 페이지에서 발견된 링크 일부:")

    count = 0

    for link in links:

        text = link.text.strip()

        if text:

            print("-", text[:100])

            count += 1

            if count >= 20:
                break


# ============================================================
# 7. 브라우저 유지
# ============================================================

print("\n===================================")
print("테스트가 끝났습니다.")
print("Chrome 창을 직접 확인하세요.")
print("===================================")

input("확인했으면 Enter를 누르세요.")

driver.quit()

print("Chrome 종료 완료")

Chrome 실행 성공!
네이버 접속 성공!
페이지 제목: NAVER
현재 URL: https://www.naver.com/

검색 완료
현재 URL: https://search.naver.com/search.naver?where=nexearch&sm=top_hty&fbm=0&ie=utf8&query=%EB%85%B8%EB%B2%A8%EB%AC%B8%ED%95%99%EC%83%81&ackey=m2sl4yq8
페이지 제목: 노벨문학상 : 네이버 검색

뉴스 검색 페이지로 이동합니다.
뉴스 페이지 접속 완료
현재 URL: https://search.naver.com/search.naver?where=news&query=%EB%85%B8%EB%B2%A8%EB%AC%B8%ED%95%99%EC%83%81
페이지 제목: 노벨문학상 : 네이버 뉴스검색

찾은 뉴스 기사 수: 0

뉴스 기사를 찾지 못했습니다.

현재 페이지에서 발견된 링크 일부:
- NAVER
- 한글 입력기
- 로그인
- 서비스 더보기
- 전체
- 뉴스
- AI
new
- 이미지
- 블로그
- 클립
- 카페
- 지식iN
- 동영상
- 쇼핑
- 어학사전
- 더보기
- 공유
- 옵션
- 관련도순
- 최신순

테스트가 끝났습니다.
Chrome 창을 직접 확인하세요.
Chrome 종료 완료


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import urllib.parse
import time


# ============================================================
# 기본 설정
# ============================================================

KEYWORD = "노벨문학상"
TARGET_COUNT = 100

print("=" * 60)
print("네이버 뉴스 동적 크롤링 시작")
print("=" * 60)
print("검색어:", KEYWORD)
print("목표 기사 수:", TARGET_COUNT)


# ============================================================
# Chrome 실행
# ============================================================

options = webdriver.ChromeOptions()

# Chrome 화면을 직접 볼 수 있도록 설정
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 15)


# ============================================================
# 검색어 URL 인코딩
# ============================================================

encoded_keyword = urllib.parse.quote(KEYWORD)


# ============================================================
# 뉴스 검색 URL
# sort=1 → 날짜순(최신순)
# ============================================================

BASE_URL = (
    "https://search.naver.com/search.naver"
    f"?where=news"
    f"&query={encoded_keyword}"
    f"&sort=1"
)


# ============================================================
# 뉴스 데이터 저장 리스트
# ============================================================

news_data = []


# ============================================================
# 1페이지 ~ 10페이지
# ============================================================

for page in range(1, 11):

    # --------------------------------------------------------
    # 네이버 뉴스 검색 페이지 번호
    # 1페이지 = start 1
    # 2페이지 = start 11
    # 3페이지 = start 21
    # ...
    # 10페이지 = start 91
    # --------------------------------------------------------

    start = (page - 1) * 10 + 1

    url = (
        f"{BASE_URL}"
        f"&start={start}"
    )

    print()
    print("-" * 60)
    print(f"[{page}/10 페이지] 접속 중...")
    print("-" * 60)

    driver.get(url)

    # 페이지 로딩 대기
    time.sleep(2)


    # --------------------------------------------------------
    # 뉴스 제목이 나타날 때까지 기다리기
    # --------------------------------------------------------

    try:

        wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "a.news_tit")
            )
        )

    except:

        print("뉴스 기사를 찾지 못했습니다.")
        continue


    # --------------------------------------------------------
    # 뉴스 기사 찾기
    # --------------------------------------------------------

    articles = driver.find_elements(
        By.CSS_SELECTOR,
        "a.news_tit"
    )

    print("발견된 기사:", len(articles))


    # --------------------------------------------------------
    # 기사 하나씩 수집
    # --------------------------------------------------------

    for article in articles:

        if len(news_data) >= TARGET_COUNT:
            break

        try:

            # ==================================================
            # 제목
            # ==================================================

            title = article.text.strip()

            if not title:
                continue


            # ==================================================
            # URL
            # ==================================================

            article_url = article.get_attribute("href")


            # ==================================================
            # 기사 전체 영역
            # ==================================================

            try:

                news_area = article.find_element(
                    By.XPATH,
                    "./ancestor::div[contains(@class,'news_area')]"
                )

            except:

                news_area = article


            # ==================================================
            # 언론사와 날짜
            # ==================================================

            press = ""
            date = ""

            try:

                info_elements = news_area.find_elements(
                    By.CSS_SELECTOR,
                    ".info_group .info"
                )

                info_texts = []

                for info in info_elements:

                    text = info.text.strip()

                    if text:
                        info_texts.append(text)


                # ----------------------------------------------
                # 언론사
                # ----------------------------------------------

                if len(info_texts) >= 1:
                    press = info_texts[0]


                # ----------------------------------------------
                # 날짜
                # ----------------------------------------------

                if len(info_texts) >= 2:

                    date = info_texts[-1]


            except:

                pass


            # ==================================================
            # 기사 요약
            # ==================================================

            description = ""

            try:

                description = news_area.find_element(
                    By.CSS_SELECTOR,
                    ".news_dsc"
                ).text.strip()

            except:

                pass


            # ==================================================
            # 중복 확인
            # ==================================================

            duplicate = any(
                item["URL"] == article_url
                for item in news_data
            )

            if duplicate:
                continue


            # ==================================================
            # 데이터 저장
            # ==================================================

            news_data.append({

                "순위": len(news_data) + 1,

                "페이지": page,

                "제목": title,

                "언론사": press,

                "날짜": date,

                "요약": description,

                "URL": article_url

            })


            print(
                f"{len(news_data):3d}. {title}"
            )


        except Exception as e:

            print("기사 처리 중 오류:", e)

            continue


    # ========================================================
    # 100건 완료 확인
    # ========================================================

    if len(news_data) >= TARGET_COUNT:

        print()
        print("🎉 100건 수집 완료!")
        break


# ============================================================
# 브라우저 종료
# ============================================================

driver.quit()


# ============================================================
# DataFrame 생성
# ============================================================

df = pd.DataFrame(news_data)


# 정확히 100건만 사용
df = df.head(TARGET_COUNT)


# ============================================================
# 결과 출력
# ============================================================

print()
print("=" * 60)
print("크롤링 완료")
print("=" * 60)

print("총 수집 기사:", len(df))


# ============================================================
# 데이터 확인
# ============================================================

display(df)


# ============================================================
# CSV 저장
# ============================================================

csv_filename = "네이버_뉴스_노벨문학상_최신순_100건.csv"

df.to_csv(
    csv_filename,
    index=False,
    encoding="utf-8-sig"
)

print()
print("CSV 저장 완료:")
print(csv_filename)


# ============================================================
# Excel 저장
# ============================================================

excel_filename = "네이버_뉴스_노벨문학상_최신순_100건.xlsx"

df.to_excel(
    excel_filename,
    index=False
)

print()
print("Excel 저장 완료:")
print(excel_filename)

네이버 뉴스 동적 크롤링 시작
검색어: 노벨문학상
목표 기사 수: 100

------------------------------------------------------------
[1/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

------------------------------------------------------------
[2/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

------------------------------------------------------------
[3/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

------------------------------------------------------------
[4/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

------------------------------------------------------------
[5/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

------------------------------------------------------------
[6/10 페이지] 접속 중...
------------------------------------------------------------
뉴스 기사를 찾지 못했습니다.

----

""



CSV 저장 완료:
네이버_뉴스_노벨문학상_최신순_100건.csv

Excel 저장 완료:
네이버_뉴스_노벨문학상_최신순_100건.xlsx


### [참고]  Selenium을 사용하여 동적 웹 페이지와 상호작용하기

* **(클릭 이벤트를 위한 xpath 복사)작업 순서**
    - 크롬에서 target 페이지 접속(https://www.naver.com/)
    - F12 눌러 오른쪽 영역에 개발자 페이지 나타나도록 함(html코드 나타남)
    - ctrl+shift+c 누른 상태에서 클릭 이벤트 발생할 곳 찾아 마우스 클릭
    - 해당 html코드 영역에서 마우스 오른쪽키 누르고 copy>copy.xpath 메뉴 선택하여 이벤트 코드 클립보드에 복사
    - driver.find_element(By.XPATH, '복사된 내용 붙여넣기').click()

* **[사용방법] 버튼(링크) 클릭**

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By

# 드라이버 초기화
driver = webdriver.Chrome()

# 웹페이지로 이동
driver.get('https://www.naver.com/')

# 클릭(copy.xpath 이용)  //*[@id="search-btn"]
#search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]')
#search_button.click()
search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]')
driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

#### [1단계] 네이버 메인페이지에서 검색어 입력하고 버튼 클릭하기

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# chrome driver를 자동으로 설치함
chromedriver_autoinstaller.install()

# 드라이버 초기화
driver = webdriver.Chrome()

def naver_main_search(driver, keyword):
    driver.get('https://www.naver.com/') # 웹페이지 로드
    search_box = driver.find_element(By.XPATH, '//*[@id="query"]')  # 검색 키워드 영역
    search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]') # 검색 버튼
    search_keyword = keyword  # 키워드
    search_box.send_keys(search_keyword)
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

# 1.네이버 메인 검색
keyword = '노벨문학상'
naver_main_search(driver, keyword)
print(f'현재URL : {driver.current_url}')


#### [2단계] 네이버 검색 결과 페이지에서 다시 버튼 클릭
- 버튼 클릭 위치 확인(xPath) : 마우스오른쪽버튼 > Copy >Copy xPath

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By

# chrome driver를 자동으로 설치함
chromedriver_autoinstaller.install()

# 드라이버 초기화
driver = webdriver.Chrome()

# [CODE 1] : 검색어 넣고 네이버 메인 검색
def naver_main_search(driver, keyword):
    print('\n1단계 : 검색어 넣고 네이버 메인 검색......')
    driver.get('https://www.naver.com/') # 웹페이지 로드
    search_box = driver.find_element(By.XPATH, '//*[@id="query"]')  # 검색 키워드 영역
    search_button = driver.find_element(By.XPATH, '//*[@id="search-btn"]') # 검색 버튼
    search_keyword = keyword  # 키워드
    search_box.send_keys(search_keyword) # 검색창에 검색어 반영
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭

# [CODE 2] : 검색 결과에서 다른 탭 선택
def naver_main_search_tab(driver, url, xpath=None, tab_name='뉴스'):
    print('\n2단계 : 검색 결과에서 탭 선택......')
    print(f'      currnet_url={driver.current_url}')
    driver.get(url) # 해당 웹페이지 로드
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '#lnb a')))

    tab_links = driver.find_elements(By.CSS_SELECTOR, '#lnb a')
    search_button = next((link for link in tab_links if tab_name in link.text or 'where=news' in (link.get_attribute('href') or '')), None)
    if search_button is None and xpath:
        search_button = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
    if search_button is None:
        raise Exception(f'{tab_name} 탭을 찾을 수 없습니다.')
    driver.execute_script('arguments[0].click();', search_button)   # 자바스크립트로 클릭
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'ul.list_news._infinite_list')))


keyword = input('페이지 검색어 입력: ')

# 1.[CODE 1] : 검색어 넣고 네이버 메인 검색
naver_main_search(driver, keyword)


# 2.[CODE 2] : 검색 결과에서 다른 탭 선택 ( 마우스오른쪽버튼 > Copy >Copy xPath)
naver_main_search_tab(driver, driver.current_url, '//*[@id="lnb"]/div[1]/div/div[1]/div[3]/a' )

-------------------------

#### [실습]  커피빈매장 정보 크롤링하여 파일로 저장하기
- 아래 사이트를 이용해 호출해야할 자바스크립트 함수를 확인하다.
- https://www.coffeebeankorea.com
- https://www.coffeebeankorea.com/store/store.asp
- (매장 번호로) 자세히보기: javascript:storePop2('374');
- chromedriver.exe 파일 위치는 코드와 동일한 위치에 놓는다.

In [ ]:
from bs4 import BeautifulSoup
import urllib.request
import pandas as pd
import datetime

from selenium import webdriver
import time

MAX = 10     # 추출 데이터 건수
FILE = './CoffeeBean_매장정보.csv'

#[CODE 1]
def getStoreInfo():
    CoffeeBean_URL = "https://www.coffeebeankorea.com/store/store.asp"

    # 드라이버 초기화
    driver = webdriver.Chrome()

    result = []  # 데이터 저장 변수
    total, cnt = 370, 0
    for i in range(1, total+1):  #매장 수 만큼(370) 반복
        driver.get(CoffeeBean_URL)
        time.sleep(1)  #웹페이지 연결할 동안 1초 대기
        try:
            print(f'read[{i}]')
            driver.execute_script("storePop2(%d)" %i)
            time.sleep(1) #스크립트 실행 할 동안 1초 대기

            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            store_name_h2 = soup.select("div.store_txt > h2")
            store_name = store_name_h2[0].string  #매장 이름

            store_info = soup.select("div.store_txt > table.store_table > tbody > tr > td")
            store_address_list = list(store_info[2])
            store_address = store_address_list[0]  #매장 주소

            store_phone = store_info[3].string     #매장 전화번호
            result.append([store_name]+[store_address]+[store_phone])
            cnt += 1
            # 매장정보 가져온 데이터 출력하기
            print("save[%3d] %3d - %s" % (cnt, i, store_name))

             # MAX값에 해당하는 건수 만큼만 실행하기
            if cnt >= MAX:
                break

        except:
            continue

    return result

#---------------
# main
#---------------
#[CODE 0]
def main():
    result = []
    print('CoffeeBean store crawling >>>>>>>>>>>>>>>>>>>>>>>>>>')
    result = getStoreInfo()  #[매장 추출 함수]호출하기   #[CODE 1] 호출
    coffeebean_tbl = pd.DataFrame(result, columns=('store', 'address','phone'))
    coffeebean_tbl.to_csv(FILE, encoding='cp949', mode='w', index=True)  # 파일로 저장하기
    del result[:]
    return coffeebean_tbl


df = main() #[CODE 0] 호출
df.head()

driver.quit()


----------------------